In [37]:
import pandas as pd
pd.set_option('display.float_format', lambda x: f"{x:.2f}")
import matplotlib.pyplot as plt

### Import data 

In [2]:
revenues_per_day = pd.read_csv('../data/revenues_per_day (1).csv')
revenues_per_day.head()

,id,date,title,revenue,theaters,distributor
0,8b19ad43-3a7e-b14b-49e9-1f7a0eb1568e,2004-09-20,Sky Captain and the World of Tomorrow,925482,3170.00,Paramount Pictures
1,481fc700-fcdd-1919-c53c-09fcd423a596,2004-09-20,Resident Evil: Apocalypse,643680,3284.00,Screen Gems
2,06719cc2-c05e-8c0b-56b6-b8b13e056509,2004-09-20,Mr 3000,425375,2736.00,Walt Disney Studios Motion Pictures
3,3be8e4a8-2716-00be-80be-eb342ba3cbe7,2004-09-20,Wimbledon,416970,2034.00,Universal Pictures
4,c5f26b59-6a64-ae8e-8f58-7cefb09eadfe,2004-09-20,Cellular,412000,2749.00,New Line Cinema


### Data Overview

#### Size

In [3]:
shape = revenues_per_day.shape
print(f'Rows = {shape[0]}, columns = {shape[1]}')

Rows = 337818, columns = 6


#### Types

In [4]:
revenues_per_day.dtypes

id                 str
date               str
title              str
revenue          int64
theaters       float64
distributor        str
dtype: object

Notes: 
* 'date' column should be a datetime object
* 'theaters' column can be an integer - check 

In [36]:
frac = (revenues_per_day['theaters'].dropna() % 1).max()
print(frac)

0.0


Notes: 'theaters' can be an integer

#### Missing values

In [5]:
revenues_per_day.isna().sum()

id               0
date             0
title            0
revenue          0
theaters       161
distributor    225
dtype: int64

#### Missing theaters with daily revenue greater than 0

In [106]:
print(f'Amount of theaters: {((df['theaters'].isna()) & (df['revenue']>0)).sum()}')

Amount of theaters: 159


#### Unique values per column

In [18]:
for col in revenues_per_day.columns:
    print(f'Is column "{col}" unique ? -> {revenues_per_day[col].is_unique}')

Is column "id" unique ? -> True
Is column "date" unique ? -> False
Is column "title" unique ? -> False
Is column "revenue" unique ? -> False
Is column "theaters" unique ? -> False
Is column "distributor" unique ? -> False


#### Number of unique values per column

In [19]:
for col in revenues_per_day.columns:
    print(f'Number of unique values in "{col}": {len(revenues_per_day[col].unique())}')

Number of unique values in "id": 337818
Number of unique values in "date": 8455
Number of unique values in "title": 6545
Number of unique values in "revenue": 179113
Number of unique values in "theaters": 4115
Number of unique values in "distributor": 363


#### Duplicates 

In [34]:
cols = ['date', 'title', 'revenue', 'theaters', 'distributor']
print(f'Duplicate rows (ignore id): {revenues_per_day.duplicated(subset=cols).sum()}')

Duplicate rows (ignore id): 0


In [35]:
cols = ['date', 'title', 'distributor']
print(f'Duplicate rows for (date, title, distributor): {revenues_per_day.duplicated(subset=cols).sum()}')

Duplicate rows for (date, title, distributor): 0


In [95]:
cols = ['date', 'title']
print(f'Duplicate rows for (date, title): {revenues_per_day.duplicated(subset=cols).sum()}')

Duplicate rows for (date, title): 0


#### Date range

In [9]:
dates = pd.to_datetime(revenues_per_day['date'])
print(f'Min date: {dates.min()}')
print(f'Max date: {dates.max()}')

Min date: 2000-01-01 00:00:00
Max date: 2023-03-06 00:00:00


In [102]:
missing = pd.date_range(dates.min(), dates.max(), freq='D').difference(pd.DatetimeIndex(dates.unique()))
print(f'Amount of missing days: {len(missing)}')
print(f'Missing days: {missing}')

Amount of missing days: 11
Missing days: DatetimeIndex(['2020-03-28', '2020-03-29', '2020-03-30', '2020-03-31',
               '2020-04-01', '2020-04-02', '2020-12-13', '2021-05-13',
               '2021-09-29', '2021-10-04', '2021-10-18'],
              dtype='datetime64[us]', freq=None)


#### Distribution in 'revenue' and 'theaters'

In [12]:
revenues_per_day.describe()

,revenue,theaters
count,337818.00,337657.00
mean,607164.20,848.61
std,2323202.56,1137.44
min,0.00,1.00
25%,6592.00,45.00
50%,40775.00,226.00
75%,302920.50,1402.00
max,157461641.00,4931.00


In [43]:
print(f"Number of revenue per day less than 0: {(revenues_per_day['revenue'] < 0).sum()}")
print(f"Number of theaters less than or equal to 0: {(revenues_per_day['theaters'].dropna() <= 0).sum()}")

Number of revenue per day less than 0: 0
Number of theaters less than or equal to 0: 0


#### Empty strings in 'title' and 'distributor'

In [55]:
for col in ['title', 'distributor']:
    print(f'Number of empty strings in "{col}": {(revenues_per_day[col].str.strip() == '').sum()}')

Number of empty strings in "title": 0
Number of empty strings in "distributor": 0


#### String normalization check

In [61]:
title_raw = revenues_per_day['title'].nunique()
title_clean = revenues_per_day['title'].str.strip().str.lower().nunique()
print(f'Number of unique titles before normalization: {title_raw}, after normalization: {title_clean}')


distributor_raw = revenues_per_day['distributor'].nunique(dropna=True)
distributor_clean = revenues_per_day['distributor'].str.strip().str.lower().nunique(dropna=True)
print(f'Number of uniqe distributors before normalization: {distributor_raw}, after normalization: {distributor_clean}')

Number of unique titles before normalization: 6545, after normalization: 6544
Number of uniqe distributors before normalization: 362, after normalization: 362


Notes: Normalizing title reduces the number of unique titles from 6545 to 6544. distributor uniqueness is unchanged.

#### Title normalization collisions

In [92]:
title_clean = revenues_per_day['title'].astype('string').str.strip().str.lower()

dup_keys = revenues_per_day.groupby(title_clean)['title'].nunique()
dup_keys = dup_keys[dup_keys > 1].index

revenues_per_day.loc[title_clean.isin(dup_keys), 'title'].drop_duplicates().sort_values()

322503    The Loneliest Whale: The Search for 52
322538    The Loneliest Whale: the Search for 52
Name: title, dtype: str

#### Top 20 titles  

In [62]:
revenues_per_day['title'].value_counts().head(20)

title
Deep Sea                                             1114
Born to Be Wild                                       822
Under the Sea 3D                                      695
Island of Lemurs: Madagascar                          533
Hubble 3D                                             502
To the Arctic 3D                                      441
NASCAR: The IMAX Experience                           371
Winter's Bone                                         300
Gladiator                                             248
The Lord of the Rings: The Two Towers                 246
The War with Grandpa                                  244
The Lord of the Rings: The Fellowship of the Ring     243
Little Miss Sunshine                                  240
The Intouchables                                      238
Avatar                                                234
Life of Pi                                            233
Bella                                                 233
Moulin R

#### Top 20 distributors

In [64]:
revenues_per_day['distributor'].value_counts(dropna=False).head(20)

distributor
Warner Bros.                           41374
Twentieth Century Fox                  28529
Universal Pictures                     27273
Walt Disney Studios Motion Pictures    18283
Paramount Pictures                     18206
Sony Pictures Entertainment (SPE)      16900
Lionsgate                              16589
Fox Searchlight Pictures               13760
Roadside Attractions                   10728
The Weinstein Company                  10120
Focus Features                          9307
-                                       7419
A24                                     4053
DreamWorks Distribution                 3956
DreamWorks                              3794
Screen Gems                             3766
Metro-Goldwyn-Mayer (MGM)               3603
Paramount Classics                      3333
Rialto Pictures                         3064
New Line Cinema                         2994
Name: count, dtype: int64

Notes: The value "-" appears to be a placeholder for missing distributor information.

#### Number of distinct distributors per title (top 20)

In [93]:
(revenues_per_day.groupby('title')['distributor']
 .nunique(dropna=True)
 .sort_values(ascending=False)
 .head(20))

title
Last Call               3
Deliver Us from Evil    2
Beast                   2
Frozen                  2
Infinity Pool           2
Emily                   2
Little Women            2
Beautiful Boy           2
Beautiful Creatures     2
Halloween               2
Bloody Hell             2
Blacklight              2
Brothers                2
Medieval                2
Iris                    2
Anna                    2
Belle                   2
Gold                    2
Les Misu00e9rables      2
Death at a Funeral      2
Name: distributor, dtype: int64

#### Unknown distributors

In [112]:
print('Revenue and theaters distribution for distributor = "-": ')
revenues_per_day.loc[revenues_per_day['distributor'].str.strip()=='-', ['revenue','theaters']].describe()

Revenue and theaters distribution for distributor = "-": 


,revenue,theaters
count,7419.00,7418.00
mean,149847.04,380.57
std,618876.47,759.09
min,0.00,1.00
25%,1381.50,10.00
50%,6616.00,51.00
75%,40279.50,241.00
max,19017001.00,3674.00


In [116]:
print('Revenue and theaters distribution for missing distributor: ')
revenues_per_day.loc[revenues_per_day['distributor'].isna(), ['revenue','theaters']].describe()

Revenue and theaters distribution for missing distributor: 


,revenue,theaters
count,225.00,222.00
mean,57602.81,178.34
std,182265.86,477.22
min,6.00,1.00
25%,273.00,1.00
50%,835.00,1.00
75%,11368.00,30.00
max,1419333.00,2212.00


# SUMMARY 

* The file contains 337,818 rows and 6 columns

* Date range is from 2000-01-01 to 2023-03-06

* There are 8,455 unique days and 11 missing calendar days inside the range

* Columns are id, date, title, distributor, revenue, theaters

* id is unique

* No duplicate rows when comparing all columns except id

* No duplicates for the keys date and title, and also date, title, distributor

* revenue has no negative values and zeros appear only rarely

* theaters has 161 missing values, and in 159 of those cases revenue is still greater than 0

* distributor has 225 missing values and the value "-" appears very often as a placeholder


* There are no empty strings in title and distributor

* There are about 6,545 unique titles and about 362 unique distributors excluding missing values

* After basic title cleaning strip and lower, the number of unique titles drops by 1

# Next steps

* Convert date to a proper DATE type

* Set final types revenue as integer and theaters as integer allowing NULL

* Standardize title and distributor trim whitespace and use consistent casing

* Map distributor equal to "-" and missing values to a single Unknown value

* Define a rule for theaters NULL when revenue is greater than 0 keep NULL and add a data quality flag or handle in a separate imputation step

* Add data quality tests in the pipeline grain uniqueness non negative revenue expected ranges

* Consider a date dimension and decide how to handle missing days in the timeline